## Download JPEGs and Calculate Extents

Prepare all photos in a selected subset for georeferencing.

- Download original JPEGs
- Read image dimensions and DPI
- Calculate approximate horizontal ground extents
- Save preparation results to a separate tracker GeoPackage

Existing valid JPEGs are reused rather than downloaded again.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
from datetime import datetime, timezone
from tqdm.auto import tqdm
import shutil

from single_photo_workflow import (
    download_photo,
    get_image_metadata,
    calculate_horizontal_extent,
    update_tracker
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

c:\Users\eu01251115\LocalProjects\batch_georef\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load subset table and make paths

In [ ]:
SUBSET_PATH = Path(
    "data/historic_airphoto_index/subsets/"
    "subset_20000_carver_37_38_40_51_57_63.gpkg"
)

LAYER = "photos"

# Create a separate GeoPackage for tracking processing progress
TRACKER_PATH = SUBSET_PATH.with_name(
    f"{SUBSET_PATH.stem}_tracker.gpkg"
)

# define subfolder for downloading images, using the geopackage name
DOWNLOAD_DIR = Path("data/downloads") / SUBSET_PATH.stem

### Make tracker geopackage and download directory

In [ ]:
# Create tracker gpkg, ONLY if it doesn't already exist
# This is a copy of the subset table, which is preserved blank for input. 

if not TRACKER_PATH.exists():
    shutil.copy2(SUBSET_PATH, TRACKER_PATH)
    print(f"Created tracker: {TRACKER_PATH.name}")
else:
    print(f"Using existing tracker: {TRACKER_PATH.name}")

Using existing tracker: subset_20000_carver_37_38_40_51_57_63_tracker.gpkg


In [ ]:
# Make download directory
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

### Load tracker, select unfinished photos

In [ ]:
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

In [ ]:
# Only prepare photos that haven't already been prepared
photos_gdf = tracker_gdf.loc[
    ~tracker_gdf["status"].isin(["prepared", "georeferenced"]) # status is NOT prepared or georeferenced
].copy()

print(f"Total photos in subset: {len(tracker_gdf)}")
print(f"Photos remaining: {len(photos_gdf)}")
print(f"Download directory: {DOWNLOAD_DIR}")

display(photos_gdf.head())

### Download JPEG, read metadata, calculate extent

In [ ]:
results = []

for _, photo in tqdm(
    batch_gdf.iterrows(),
    total=len(batch_gdf),
    desc="Preparing photos"
):
    photo_id = photo["cty_year_id"]

    image_path = DOWNLOAD_DIR / f"{photo_id}_original.jpg"

    result = {
        "source_id": photo["source_id"],
        "cty_year_id": photo_id,
        "width_px": None,
        "dpi_x": None,
        "h_extent": None,
        "date_download": None,
        "download_status": "pending",
        "error": None,
    }

    try:
        image_path = download_photo(
            url=photo["url"],
            output_path=image_path,
        )

        result["download_status"] = "complete"

        result["date_download"] = datetime.now(
            timezone.utc
        ).isoformat()

        metadata = get_image_metadata(image_path)

        result.update(metadata)

        result["h_extent"] = calculate_horizontal_extent(
            width_px=metadata["width_px"],
            dpi_x=metadata["dpi_x"],
            scale=photo["scale"],
        )

    except Exception as e:
        result["error"] = str(e)

    results.append(result)

Preparing photos: 100%|██████████| 100/100 [00:00<00:00, 611.91it/s]


In [ ]:
# Update the tracker table
updated_count = update_tracker(
    TRACKER_PATH,
    results_df,
    layer=LAYER,
)

print(f"Updated {updated_count} records.")

,cty_year_id,status,county,year,photo_id,h_extent,gcp_count,confidence,scale,dpi_x,width_px,worldfile_json,url,flight_code,source_id,date_download,date_georef,geometry
0,carver_1937_wj-4-172,prepared,Carver,1937,wj-4-172,4529.455,None,None,20000,800,7133,None,https://geo.lib.umn.edu/scott/y1937/wj-4-172.jpg,Scott,96681,2026-10-21T20:54:17.617331+00:00,None,POINT (-10413310.856 5589608.567)
1,carver_1937_wj-4-173,prepared,Carver,1937,wj-4-173,4526.915,None,None,20000,800,7129,None,https://geo.lib.umn.edu/scott/y1937/wj-4-173.jpg,Scott,96682,2026-10-21T20:54:17.618915+00:00,None,POINT (-10413334.456 5591176.036)
2,carver_1937_wj-4-174,prepared,Carver,1937,wj-4-174,4533.265,None,None,20000,800,7139,None,https://geo.lib.umn.edu/scott/y1937/wj-4-174.jpg,Scott,96683,2026-10-21T20:54:17.620235+00:00,None,POINT (-10413358.166 5592743.776)
3,carver_1937_wj-5-177,prepared,Carver,1937,wj-5-177,4512.945,None,None,20000,800,7107,None,https://geo.lib.umn.edu/scott/y1937/wj-5-177.jpg,Scott,96638,2026-10-21T20:54:17.621486+00:00,None,POINT (-10417918.258 5591066.676)
4,carver_1937_wj-5-178,prepared,Carver,1937,wj-5-178,4533.265,None,None,20000,800,7139,None,https://geo.lib.umn.edu/scott/y1937/wj-5-178.jpg,Scott,96639,2026-10-21T20:54:17.622717+00:00,None,POINT (-10417893.1 5589204.938)


In [ ]:
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

print("Preparation status:")
print(tracker_gdf["status"].value_counts(dropna=False))

display(
    tracker_gdf[
        [
            "cty_year_id",
            "scale",
            "width_px",
            "dpi_x",
            "h_extent",
            "status",
        ]
    ].head(20)
)

In [16]:
updated = update_tracker(
    TRACKER_PATH,
    results_df,
    layer=LAYER,
)

print(f"Updated {updated} records in tracker.")

OperationalError: no such function: ST_IsEmpty

In [6]:
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

display(
    tracker_gdf[
        ["cty_year_id", "status", "width_px",
         "dpi_x", "h_extent", "date_download"]
    ].head()
)

,cty_year_id,status,width_px,dpi_x,h_extent,date_download
0,carver_1937_wm-26-2368,None,None,None,None,None
1,carver_1937_wm-26-2369,None,None,None,None,None
2,carver_1937_wm-26-2370,None,None,None,None,None
3,carver_1937_wm-26-2371,None,None,None,None,None
4,carver_1937_wm-26-2372,None,None,None,None,None


In [5]:
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

tracker_gdf

,cty_year_id,status,county,year,photo_id,h_extent,gcp_count,confidence,scale,dpi_x,width_px,worldfile_json,url,flight_code,source_id,date_download,date_georef,geometry
0,carver_1937_wm-26-2368,None,Carver,1937,wm-26-2368,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2368.jpg,Carver,91674,None,None,POINT (-10465155.126 5579058.785)
1,carver_1937_wm-26-2369,None,Carver,1937,wm-26-2369,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2369.jpg,Carver,91675,None,None,POINT (-10465099.911 5580868.601)
2,carver_1937_wm-26-2370,None,Carver,1937,wm-26-2370,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2370.jpg,Carver,91676,None,None,POINT (-10465044.585 5582678.778)
3,carver_1937_wm-26-2371,None,Carver,1937,wm-26-2371,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2371.jpg,Carver,91677,None,None,POINT (-10464989.371 5584489.317)
4,carver_1937_wm-26-2372,None,Carver,1937,wm-26-2372,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2372.jpg,Carver,91678,None,None,POINT (-10464934.045 5586300.061)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
866,carver_1963_wm-1dd-111,None,Carver,1963,wm-1dd-111,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-111.jpg,Carver,92355,None,None,POINT (-10428717.028 5574748.276)
867,carver_1963_wm-1dd-113,None,Carver,1963,wm-1dd-113,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-113.jpg,Carver,92356,None,None,POINT (-10424133.337 5574825.175)
868,carver_1963_wm-1dd-228,None,Carver,1963,wm-1dd-228,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-228.jpg,Carver,92362,None,None,POINT (-10435676.055 5570228.328)
869,carver_1963_wm-1dd-230,None,Carver,1963,wm-1dd-230,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-230.jpg,Carver,92363,None,None,POINT (-10431122.308 5570238.503)
